In [ ]:
import os, shutil, random, time, zipfile
from pathlib import Path
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from sklearn.metrics import confusion_matrix, accuracy_score, precision_recall_fscore_support

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
for folder in ["/content/mango_raw", "/content/mango_combined", "/content/mango_split"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)

In [ ]:
ZIP_DIR = "/content/drive/MyDrive/mango_dataset"
ZIP_FILES = [
    "MangoOriginal.zip",
    "MangoAugmented.zip",
    "MangoRealVirtual.zip"
]

RAW_DIR = "/content/mango_raw"
os.makedirs(RAW_DIR, exist_ok=True)

for z in ZIP_FILES:
    zp = f"{ZIP_DIR}/{z}"
    if os.path.exists(zp):
        print("Extracting:", z)
        with zipfile.ZipFile(zp, "r") as zip_ref:
            zip_ref.extractall(RAW_DIR)

print("Extraction complete")

Extracting: MangoOriginal.zip
Extracting: MangoAugmented.zip
Extracting: MangoRealVirtual.zip
Extraction complete


In [ ]:
COMBINED_DIR = "/content/mango_combined"
os.makedirs(COMBINED_DIR, exist_ok=True)

EXTS = (".jpg",".jpeg",".png",".bmp",".webp")

for root, dirs, files in os.walk(RAW_DIR):
    imgs = [f for f in files if f.lower().endswith(EXTS)]
    if imgs:
        class_name = os.path.basename(root).replace(" ", "_").replace("'", "")
        dest = os.path.join(COMBINED_DIR, class_name)
        os.makedirs(dest, exist_ok=True)

        for img in imgs:
            shutil.copy(os.path.join(root, img), os.path.join(dest, img))

        print(f"Class {class_name}: {len(imgs)} images")

Class Rupali: 920 images
Class Himsagor: 530 images
Class Bari-4: 370 images
Class Harivanga: 1325 images
Class Ashshina_Zhinuk: 6430 images
Class Fazli_Shurmai: 1235 images
Class Banana_Mango: 415 images
Class Langra: 600 images
Class Katimon: 2120 images
Class Amrapali: 675 images
Class Bari-11: 6220 images
Class Fazli_Classic: 855 images
Class Shada: 815 images
Class Ashshina_Classic: 2855 images
Class Gourmoti: 3150 images


In [ ]:
SPLIT_DIR = "/content/mango_split"
TRAIN_DIR = f"{SPLIT_DIR}/train"
VAL_DIR   = f"{SPLIT_DIR}/val"

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

random.seed(42)

for cls in os.listdir(COMBINED_DIR):
    imgs = os.listdir(os.path.join(COMBINED_DIR, cls))
    random.shuffle(imgs)

    split = int(0.8 * len(imgs))
    train_imgs, val_imgs = imgs[:split], imgs[split:]

    os.makedirs(f"{TRAIN_DIR}/{cls}", exist_ok=True)
    os.makedirs(f"{VAL_DIR}/{cls}", exist_ok=True)

    for img in train_imgs:
        shutil.copy(f"{COMBINED_DIR}/{cls}/{img}", f"{TRAIN_DIR}/{cls}/{img}")
    for img in val_imgs:
        shutil.copy(f"{COMBINED_DIR}/{cls}/{img}", f"{VAL_DIR}/{cls}/{img}")

    print(cls, f"train={len(train_imgs)} val={len(val_imgs)}")

Rupali train=736 val=184
Himsagor train=424 val=106
Bari-4 train=296 val=74
Ashshina_Classic train=2284 val=571
Harivanga train=1060 val=265
Langra train=480 val=120
Katimon train=1696 val=424
Banana_Mango train=332 val=83
Ashshina_Zhinuk train=5144 val=1286
Fazli_Shurmai train=988 val=247
Amrapali train=540 val=135
Bari-11 train=4976 val=1244
Shada train=652 val=163
Fazli_Classic train=684 val=171
Gourmoti train=2520 val=630


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_ds = datasets.ImageFolder(TRAIN_DIR, train_tf)
val_ds   = datasets.ImageFolder(VAL_DIR, val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)

class_names = train_ds.classes
num_classes = len(class_names)
print("Classes:", class_names)

Using device: cpu
Classes: ['Amrapali', 'Ashshina_Classic', 'Ashshina_Zhinuk', 'Banana_Mango', 'Bari-11', 'Bari-4', 'Fazli_Classic', 'Fazli_Shurmai', 'Gourmoti', 'Harivanga', 'Himsagor', 'Katimon', 'Langra', 'Rupali', 'Shada']


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU()
        )
    def forward(self, x): return self.net(x)

class ResidualDownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.c1 = ConvBlock(in_ch, out_ch, 2)
        self.c2 = ConvBlock(out_ch, out_ch)
        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, 2, bias=False),
            nn.BatchNorm2d(out_ch)
        )
    def forward(self, x):
        return torch.relu(self.c2(self.c1(x)) + self.skip(x))

class CustomCNN(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.stem = nn.Sequential(
            ConvBlock(3,32),
            ConvBlock(32,32),
            nn.MaxPool2d(2)
        )
        self.s1 = ResidualDownBlock(32,64)
        self.s2 = ResidualDownBlock(64,128)
        self.s3 = ResidualDownBlock(128,256)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256,n)

    def forward(self,x):
        x = self.stem(x)
        x = self.s1(x)
        x = self.s2(x)
        x = self.s3(x)
        return self.fc(self.pool(x).flatten(1))

In [ ]:
def train_model(model, name):
    model.to(device)
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    tr_acc, va_acc, tr_loss, va_loss = [], [], [], []
    start = time.time()

    for epoch in range(1, EPOCHS+1):
        print(f"\n[{name}] Epoch {epoch}/{EPOCHS}")

        # Train
        model.train()
        correct = total = loss_sum = 0
        for x,y in tqdm(train_loader, desc="Training"):
            x,y = x.to(device), y.to(device)
            opt.zero_grad()
            out = model(x)
            loss = criterion(out,y)
            loss.backward()
            opt.step()

            loss_sum += loss.item()
            correct += (out.argmax(1)==y).sum().item()
            total += y.size(0)

        tr_acc.append(correct/total)
        tr_loss.append(loss_sum/len(train_loader))

        # Validation
        model.eval()
        correct = total = loss_sum = 0
        with torch.no_grad():
            for x,y in tqdm(val_loader, desc="Validating"):
                x,y = x.to(device), y.to(device)
                out = model(x)
                loss = criterion(out,y)

                loss_sum += loss.item()
                correct += (out.argmax(1)==y).sum().item()
                total += y.size(0)

        va_acc.append(correct/total)
        va_loss.append(loss_sum/len(val_loader))

        print(f"Train Acc={tr_acc[-1]:.3f} | Val Acc={va_acc[-1]:.3f}")

    return tr_acc, va_acc, tr_loss, va_loss, time.time()-start

In [ ]:
custom = CustomCNN(num_classes)
c_tr_acc, c_va_acc, c_tr_loss, c_va_loss, c_time = train_model(custom, "Custom CNN")

# ResNet Scratch
res_s = models.resnet18(weights=None)
res_s.fc = nn.Linear(res_s.fc.in_features, num_classes)
rs_tr_acc, rs_va_acc, rs_tr_loss, rs_va_loss, rs_time = train_model(res_s, "ResNet Scratch")

# VGG Scratch
vgg_s = models.vgg16(weights=None)
vgg_s.classifier[6] = nn.Linear(4096, num_classes)
vs_tr_acc, vs_va_acc, vs_tr_loss, vs_va_loss, vs_time = train_model(vgg_s, "VGG Scratch")

resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in resnet.parameters(): p.requires_grad=False
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
r_tr_acc, r_va_acc, r_tr_loss, r_va_loss, r_time = train_model(resnet, "ResNet-18 TL")

vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
for p in vgg.features.parameters(): p.requires_grad=False
vgg.classifier[6] = nn.Linear(4096, num_classes)
v_tr_acc, v_va_acc, v_tr_loss, v_va_loss, v_time = train_model(vgg, "VGG-16 TL")


[Custom CNN] Epoch 1/5


Training:   0%|          | 0/713 [00:00<?, ?it/s]

Validating:   0%|          | 0/179 [00:00<?, ?it/s]

Train Acc=0.729 | Val Acc=0.779

[Custom CNN] Epoch 2/5


Training:   0%|          | 0/713 [00:00<?, ?it/s]

...

Train Acc=0.904 | Val Acc=0.537

[Custom CNN] Epoch 5/5


Training:   0%|          | 0/713 [00:00<?, ?it/s]

Validating:   0%|          | 0/179 [00:00<?, ?it/s]

Train Acc=0.917 | Val Acc=0.315

[ResNet Scratch] Epoch 1/5


Training:   0%|          | 0/713 [00:00<?, ?it/s]

In [ ]:
# ResNet Scratch
res_s = models.resnet18(weights=None)
res_s.fc = nn.Linear(res_s.fc.in_features, num_classes)
rs_tr_acc, rs_va_acc, rs_tr_loss, rs_va_loss, rs_time = train_model(res_s, "ResNet Scratch")


[ResNet Scratch] Epoch 1/5


Training:   0%|          | 0/713 [00:00<?, ?it/s]

Validating:   0%|          | 0/179 [00:00<?, ?it/s]

Train Acc=0.729 | Val Acc=0.782

[ResNet Scratch] Epoch 2/5


Training:   0%|          | 0/713 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/179 [00:00<?, ?it/s]

Train Acc=0.900 | Val Acc=0.475

[ResNet Scratch] Epoch 5/5


Training:   0%|          | 0/713 [00:00<?, ?it/s]

Validating:   0%|          | 0/179 [00:00<?, ?it/s]

Train Acc=0.914 | Val Acc=0.895


In [ ]:
# VGG Scratch
vgg_s = models.vgg16(weights=None)
vgg_s.classifier[6] = nn.Linear(4096, num_classes)
vs_tr_acc, vs_va_acc, vs_tr_loss, vs_va_loss, vs_time = train_model(vgg_s, "VGG Scratch")


[VGG Scratch] Epoch 1/5


Training:   0%|          | 0/713 [00:00<?, ?it/s]

In [ ]:
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in resnet.parameters(): p.requires_grad=False
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
r_tr_acc, r_va_acc, r_tr_loss, r_va_loss, r_time = train_model(resnet, "ResNet-18 TL")

In [ ]:
vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
for p in vgg.features.parameters(): p.requires_grad=False
vgg.classifier[6] = nn.Linear(4096, num_classes)
v_tr_acc, v_va_acc, v_tr_loss, v_va_loss, v_time = train_model(vgg, "VGG-16 TL")

In [ ]:
epochs = range(1,EPOCHS+1)

plt.figure(figsize=(8,5))
plt.plot(epochs,c_tr_acc,'--o',label="Custom Train")
plt.plot(epochs,c_va_acc,'-o',label="Custom Val")
plt.plot(epochs,r_tr_acc,'--s',label="ResNet Train")
plt.plot(epochs,r_va_acc,'-s',label="ResNet Val")
plt.plot(epochs,v_tr_acc,'--^',label="VGG Train")
plt.plot(epochs,v_va_acc,'-^',label="VGG Val")
plt.legend(); plt.grid()
plt.title("Mango Dataset – Accuracy vs Epoch")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(epochs,c_tr_loss,'--o',label="Custom Train")
plt.plot(epochs,c_va_loss,'-o',label="Custom Val")
plt.plot(epochs,r_tr_loss,'--s',label="ResNet Train")
plt.plot(epochs,r_va_loss,'-s',label="ResNet Val")
plt.plot(epochs,v_tr_loss,'--^',label="VGG Train")
plt.plot(epochs,v_va_loss,'-^',label="VGG Val")
plt.legend(); plt.grid()
plt.title("Mango Dataset – Loss vs Epoch")
plt.show()

In [ ]:
def preds(model):
    model.eval()
    y,p=[],[]
    with torch.no_grad():
        for x,yb in val_loader:
            out=model(x.to(device))
            y+=yb.tolist()
            p+=out.argmax(1).cpu().tolist()
    return y,p
for model,name in [(custom,"Custom CNN"),(resnet,"ResNet-18"),(vgg,"VGG-16")]:
    y,p = preds(model)
    plt.figure(figsize=(5,4))
    sns.heatmap(confusion_matrix(y,p),annot=True,fmt="d",
                xticklabels=class_names,yticklabels=class_names,cmap="Blues")
    plt.title(name)
    plt.show()

In [ ]:
def metrics(y,p):
    acc = accuracy_score(y,p)
    pr,rc,f1,_ = precision_recall_fscore_support(y,p,average="weighted",zero_division=0)
    return acc,pr,rc,f1

df = pd.DataFrame({
    "Model":["Custom CNN","ResNet-18 TL","VGG-16 TL"],
    "Accuracy":[metrics(*preds(custom))[0],
                metrics(*preds(resnet))[0],
                metrics(*preds(vgg))[0]],
    "Precision":[metrics(*preds(custom))[1],
                 metrics(*preds(resnet))[1],
                 metrics(*preds(vgg))[1]],
    "Recall":[metrics(*preds(custom))[2],
              metrics(*preds(resnet))[2],
              metrics(*preds(vgg))[2]],
    "F1-Score":[metrics(*preds(custom))[3],
                metrics(*preds(resnet))[3],
                metrics(*preds(vgg))[3]],
    "Training Time (s)":[c_time,r_time,v_time]
}).round(4)

df